In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv(r'C:\Dev\music4u\datasets\SpotifyAudioFeaturesApril2019.csv')

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df

,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.1180,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.3710,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.3820,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.6410,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.9280,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130658,Calum Scott,0cvfSKcm9VeduwyYPrxtLx,Come Back Home,0.006780,0.601,190539,0.801,0.000,11,0.0991,-5.174,1,0.0323,131.049,4,0.2890,57
130659,Saint Claire,43MP9F7UzvfilSrw2SqZGJ,Enough for You,0.918000,0.387,194583,0.249,0.000,9,0.1030,-13.233,1,0.0437,94.039,4,0.3460,60
130660,Mike Stud,4TWlUuFk81NGUNKwndyS5Q,Do It,0.330000,0.717,139191,0.532,0.000,8,0.0997,-8.351,0,0.2060,156.977,4,0.5460,47
130661,D Savage,5iGBXzOoRo4sBTy8wdzMyK,No Smoke,0.007900,0.772,180013,0.510,0.000,4,0.1310,-9.670,0,0.1200,120.049,4,0.0755,50


In [ ]:
df=df.dropna(subset=['track_name'])

In [ ]:
metadata_cols=['track_id','artist_name','track_name']
feature_cols=[
    'acousticness',	'danceability'	,
    'duration_ms',	'energy',
    'instrumentalness',	'liveness'	,
    'loudness',
    'speechiness',	'tempo',
    'valence'
]

In [ ]:
df=df.reset_index(drop=True)

In [ ]:
scale=MinMaxScaler()
scaled_features=scale.fit_transform(df[feature_cols])

In [ ]:
nn_model = NearestNeighbors(n_neighbors=50,metric='cosine',algorithm='brute')
nn_model.fit(scaled_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",50
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [ ]:
#find sond helper function
def find_song(query,df):
    return df[
        df['track_name'].str.contains(
            query,
            case=False,
            na=False
        )
    ][['artist_name','track_name']]
find_song('You Are In Love',df)

,artist_name,track_name
90625,Taylor Swift,You Are In Love


In [ ]:
def rec_with_nn(
    track_name,
    artist_name,
    df,
    scaled_matrix,
    model,
    metadata_cols,
    n_recommendations=10
):

    match = df[
        (df["track_name"].str.lower() == track_name.lower()) &
        (df["artist_name"].str.lower() == artist_name.lower())
    ]

    if match.empty:
        print(f"'{track_name}' by '{artist_name}' not found.")
        return None

    song_idx = match.index[0]

    query_vector = scaled_matrix[song_idx].reshape(1, -1)

    distances, indices = model.kneighbors(
        query_vector,
        n_neighbors=n_recommendations + 1
    )

    neighbor_indices = indices.flatten()[1:]
    similarity_scores = 1 - distances.flatten()[1:]

    output_df = (
        df.iloc[neighbor_indices][metadata_cols]
        .copy()
    )

    output_df["similarity_score"] = similarity_scores

    return output_df.sort_values(
        "similarity_score",
        ascending=False
    ).reset_index(drop=True)

In [ ]:
recommendations = rec_with_nn(
    track_name="You Are In Love",
    artist_name="Taylor Swift",
    df=df,
    scaled_matrix=scaled_features,
    model=nn_model,
    metadata_cols=metadata_cols
)
print(recommendations)

                 track_id           artist_name                 track_name  \
0  68XRaNESU7SGAwTRPdisKf           Alicia Keys   Un-thinkable (I'm Ready)   
1  0DOB3exHdPxofPOckSQzGN           Sherry Khan               Banke Saanse   
2  3CCIE5fXr8J5pa8Ngn1cWQ                J.BASS          Stay With Me 붙어있자   
3  3VsYGMzVwlDRCSyjCy8zCt           Tony Kakkar  Mohabbat Nasha Hai (Duet)   
4  0T7tq31vimg4GfottWxL5V     The Neighbourhood                Too Serious   
5  5yljgjCRPLBgYUrXEijTRc                 Karsu                  Jest Oldu   
6  0fbUhtnSVRFtuRAQVyRw2l            Sree Kavya     Needi Naadhi Oke Katha   
7  2i55pKLuTymvzCyBe0wXw3        Jake Etheridge  The Persistence of Memory   
8  2KfpVGOjL04HE5hoRjD57f  Steep Canyon Rangers          When She Was Mine   
9  4eJ6xxhMkNjovi7w6qQMS9             Jake Owen               Made For You   

   similarity_score  
0          0.999505  
1          0.999463  
2          0.999314  
3          0.999299  
4          0.999223  
5        